In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
data = PyMuPDFLoader("data/Attached.pdf")
data_content=data.load()

/tmp/ipykernel_124320/1254513910.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100,)
chunks=text_splitter.split_documents(data_content)


In [26]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")
GROQ_API =os.environ['GROQ_API']=os.getenv("GROQ_API")

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
hg_embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
from langchain_chroma import Chroma
Vector_DB= Chroma.from_documents(
    documents=chunks,
    embedding=hg_embedding,
    collection_name="pdf_embedding_vectorDB",
    persist_directory="Chrome_DB")

In [18]:
question="what is attachment theory"
results = Vector_DB.similarity_search(query="what is attachment theory")
context = ""
for result in results:
    context += result.page_content


In [19]:
prompt = f"""
You are a helpful AI assistant.

Your task is to answer the user's question using ONLY the information provided in the context below.

Rules:
1. Answer only from the provided context.
2. Do not make up facts or use outside knowledge.
3. If the answer is not present in the context, reply:
   "I couldn't find that information in the provided document."
4. Keep the answer clear and concise.

==================== CONTEXT ====================

{context}

=================================================

Question:
{question}

Answer:
"""


In [27]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024,api_key=GROQ_API)
response=llm.invoke(prompt)
response.content

'Attachment theory is an overarching theory of romantic affiliation that allows for the development of useful applications for people in all stages of their romantic life.'